# Colour vision deficiency

**Colour vision deficiency -- and how to check instead of hoping.**

About 1 in 12 men and 1 in 200 women see colour differently. Red-green confusion (deuteranopia and protanopia) is by far the most common. If your chart's meaning lives in the difference between red and green, those readers get nothing.

**What it shows:**

- a simulation of how a palette looks with each kind of CVD
- matplotlib's default tab10 measured -- which pairs actually collide
- Okabe-Ito measured the same way. Better, but read the numbers: it is clean for deuteranopia (the common case) and still has collisions for tritanopia (the rare one). No palette is safe for everything.
- which is exactly why colour must never be the ONLY channel

The simulation is the standard Viénot-Brettel-Mollon approximation. It is a teaching tool, not a clinical one -- but it is far better than guessing.

---

*Chapter:* `color` — palettes, colour blindness, and why rainbow lies  
*Run the cells in order.* Every figure is also written to `viz/output/color/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save()` writes each figure into `viz/output/` **and** leaves it on screen here. The trailing `;` on those calls only stops the notebook echoing the path it returns.


In [ ]:
%matplotlib inline

# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb

from vizkit import save

# Where save() files this lesson's output: viz/output/color/
LESSON = "color/colorblind"


## The simulation

The Viénot–Brettel–Mollon approximation: convert sRGB to LMS cone response, flatten the missing cone's axis, convert back. It is a teaching tool rather than a clinical one, but it beats guessing by a wide margin.


In [ ]:
RGB_TO_LMS = np.array([[17.8824, 43.5161, 4.11935],
                       [3.45565, 27.1554, 3.86714],
                       [0.0299566, 0.184309, 1.46709]])
LMS_TO_RGB = np.linalg.inv(RGB_TO_LMS)

PROJECTIONS = {
    "protanopia":   np.array([[0, 2.02344, -2.52581], [0, 1, 0], [0, 0, 1]]),
    "deuteranopia": np.array([[1, 0, 0], [0.494207, 0, 1.24827], [0, 0, 1]]),
    "tritanopia":   np.array([[1, 0, 0], [0, 1, 0], [-0.395913, 0.801109, 0]]),
}


def simulate(colour, kind):
    """One RGB colour (0-1), as a person with `kind` would see it."""
    rgb = np.array(to_rgb(colour)) * 255
    lms = RGB_TO_LMS @ rgb
    seen = LMS_TO_RGB @ (PROJECTIONS[kind] @ lms)
    return np.clip(seen / 255, 0, 1)


def confusable_pairs(palette, kind, threshold=0.18):
    """Which pairs become hard to tell apart? Distance in RGB, roughly."""
    seen = [simulate(c, kind) for c in palette]
    pairs = []
    for i in range(len(seen)):
        for j in range(i + 1, len(seen)):
            distance = float(np.linalg.norm(seen[i] - seen[j]))
            if distance < threshold:
                pairs.append((i, j, distance))
    return pairs


## 1. See the difference

Read across each row. tab10's red and green collapse onto each other for the two common kinds of CVD; Okabe-Ito was designed so they do not.


In [ ]:
TAB10 = list(plt.get_cmap("tab10").colors)[:6]
# Okabe-Ito: designed for colour vision deficiency, and widely recommended.
SAFE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00"]

fig, axes = plt.subplots(2, 4, figsize=(13, 4))

for row, (palette, name) in enumerate([(TAB10, "matplotlib tab10"),
                                       (SAFE, "Okabe-Ito (CVD-safe)")]):
    for col, kind in enumerate(["normal"] + list(PROJECTIONS)):
        ax = axes[row, col]
        # imshow wants RGB triples, not "#0072B2" strings.
        shown = ([to_rgb(c) for c in palette] if kind == "normal"
                 else [simulate(c, kind) for c in palette])
        ax.imshow([shown], aspect="auto")
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            ax.set_title(kind, fontsize=10)
        if col == 0:
            ax.set_ylabel(name, fontsize=9)

fig.suptitle("The same two palettes, as four different readers see them", fontsize=12)
fig.tight_layout()
save(fig, LESSON, "palettes-simulated");


## 2. Measure it, do not eyeball it

Eyeballing a simulation is still guessing. This measures the distance between every pair of simulated colours and names the ones that collide — note that Okabe-Ito is clean for deuteranopia and still collides for tritanopia. No palette is safe for everything.


In [ ]:
print("  colours that become hard to tell apart (RGB distance < 0.18):")
for palette, name in [(TAB10, "tab10"), (SAFE, "Okabe-Ito")]:
    print(f"\n    {name}")
    for kind in PROJECTIONS:
        pairs = confusable_pairs(palette, kind)
        if pairs:
            detail = ", ".join(f"{i}&{j} ({d:.2f})" for i, j, d in pairs)
            print(f"      {kind:<13} {len(pairs)} collision(s): {detail}")
        else:
            print(f"      {kind:<13} none")


## 3. Colour should never be the only channel

Which is the argument for this last figure. If the meaning also lives in line style, marker and a direct label, then losing the colour channel costs the reader nothing.


In [ ]:
x = np.linspace(0, 10, 100)
series = {"control": np.sin(x), "treatment": np.sin(x) + 0.6,
          "placebo": np.sin(x) - 0.6}
risky = ["#D62728", "#2CA02C", "#8C564B"]        # red / green / brown

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

for (name, values), colour in zip(series.items(), risky):
    axes[0].plot(x, values, color=colour, label=name, lw=2)
axes[0].legend(); axes[0].set_title("Colour only", fontsize=10)

for (name, values), colour in zip(series.items(), risky):
    axes[1].plot(x, values, color=simulate(colour, "deuteranopia"), label=name, lw=2)
axes[1].legend(); axes[1].set_title("...as deuteranopia sees it", fontsize=10)

for (name, values), colour, style, marker in zip(
        series.items(), SAFE, ["-", "--", ":"], ["o", "s", "^"]):
    axes[2].plot(x, values, color=colour, label=name, lw=2,
                 linestyle=style, marker=marker, markevery=15, markersize=5)
axes[2].legend(); axes[2].set_title("Colour + line style + marker", fontsize=10)

fig.suptitle("If colour is the only difference, some readers see one chart",
             fontsize=12)
fig.tight_layout()
save(fig, LESSON, "redundant-encoding");


## Rules of thumb

```text
Read those numbers again: Okabe-Ito is clean for deuteranopia -- the most
common kind -- but still collides for tritanopia. There is no palette that
is safe for every reader, which is the real argument for the third figure.

Rules:
  do not put meaning in red-vs-green alone
  use Okabe-Ito for categories, viridis for continuous data
  add a second channel: line style, marker, direct labels
  print it in greyscale -- if it still works, it works for everyone
```


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. Run `confusable_pairs` on `plt.get_cmap('tab20').colors[:10]`. How many collisions, and for which kinds?
2. Lower `threshold` from 0.18 to 0.10 and re-run section 2. Which collisions survive a stricter test?
3. Take any chart you have made this week, simulate its palette here, and print the collisions. Fix whichever one carries meaning.


In [ ]:
# your turn


---

**Previous:** [`color/palettes`](palettes.ipynb)  
**Next:** [`color/rainbow`](rainbow.ipynb)
